# Neural Pricing Surrogates & Greeks

Phase 2 reads the committed johnhull JSON/NPZ reference only; this notebook never starts training.

## Data policy and artifact fingerprint

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np

root = Path.cwd()
metrics_path = (
    root / "../johnhull/volumes/18_ml_surrogates/reference/pricing_metrics.json"
).resolve()
reference = json.loads(metrics_path.read_text(encoding="utf-8"))
companion_name = "pricing_slices.npz"
arrays_path = metrics_path.parent / companion_name
expected_sha256 = reference["companions"][companion_name]
actual_sha256 = hashlib.sha256(arrays_path.read_bytes()).hexdigest()
if actual_sha256 != expected_sha256:
    raise ValueError("pricing_slices.npz hash does not match pricing_metrics.json")
with np.load(arrays_path, allow_pickle=False) as payload:
    slices = {name: payload[name] for name in payload.files}
expected_arrays = set(reference["companion_schemas"][companion_name])
if set(slices) != expected_arrays:
    raise ValueError("pricing_slices.npz arrays do not match the committed schema")
metrics = reference["metrics"]
reference["engine_evidence"]["dataset_split_fingerprints"]

{'ood': 'sha256:d7b9931976238b936b27e47f4161b87ef92d17aa521badd1fe673a39fc46932d',
 'test': 'sha256:41a351c53ea9177ab69179a8fedd3ed91598c2bd81fe15b5b26fc74c0bb7bc38',
 'train': 'sha256:3bc76cdee74d41a42458a7f514a318b51b6b246d828347f0cb358dbbf4635b9e',
 'validation': 'sha256:7da3dd99040830561e3130d65cf805edc5cd1c40c5b2eacfe39fccfd3fd9cfb2'}

## Dimensionless Black–Scholes reference

Inputs are `x = S/K`, `τ = T`, `r`, `q`, and `σ`; output is `C/K`.

In [2]:
{
    "moneyness_range": (slices["moneyness"].min(), slices["moneyness"].max()),
    "maturity_range": (slices["maturity"].min(), slices["maturity"].max()),
}

{'moneyness_range': (np.float64(0.65), np.float64(1.35)),
 'maturity_range': (np.float64(0.03), np.float64(2.0))}

## Split and OOD audit

In [3]:
{
    "split_overlap_count": metrics["split_overlap_count"],
    "fingerprints": reference["engine_evidence"]["dataset_split_fingerprints"],
    "ood_claim": reference["limitations"][1],
}

{'split_overlap_count': 0,
 'fingerprints': {'ood': 'sha256:d7b9931976238b936b27e47f4161b87ef92d17aa521badd1fe673a39fc46932d',
  'test': 'sha256:41a351c53ea9177ab69179a8fedd3ed91598c2bd81fe15b5b26fc74c0bb7bc38',
  'train': 'sha256:3bc76cdee74d41a42458a7f514a318b51b6b246d828347f0cb358dbbf4635b9e',
  'validation': 'sha256:7da3dd99040830561e3130d65cf805edc5cd1c40c5b2eacfe39fccfd3fd9cfb2'},
 'ood_claim': 'The OOD shell remains a stress diagnostic and is not an acceptance claim.'}

## Neural price and Greek diagnostics

In [4]:
{
    "price_mae_normalized": metrics["price_mae_normalized"],
    "delta_mae": metrics["delta_mae"],
    "slice_price_error_max": slices["price_error"].max(),
    "slice_delta_error_mean": slices["delta_error"].mean(),
    "slice_gamma_error_mean": slices["gamma_error"].mean(),
}

{'price_mae_normalized': 0.0005593846268355811,
 'delta_mae': 0.0017494819891811248,
 'slice_price_error_max': np.float64(0.0009220675889730634),
 'slice_delta_error_mean': np.float64(0.0024763323647558856),
 'slice_gamma_error_mean': np.float64(0.139632603303235)}

## Differential ML and residual correction

Loss weights are checkpointed. A negative result retains the simpler price-only baseline.

In [5]:
{
    "dml_improved_a_greek_without_price_degradation": metrics[
        "dml_improved_a_greek_without_price_degradation"
    ],
    "heston_raw_price_mae": metrics["heston_raw_price_mae"],
    "heston_bsm_residual_mae": metrics["heston_bsm_residual_mae"],
}

{'dml_improved_a_greek_without_price_degradation': True,
 'heston_raw_price_mae': 0.012620526620790065,
 'heston_bsm_residual_mae': 0.0006117290429284944}

## Hard arbitrage diagnostics

In [6]:
{
    "hard_violation_rate": metrics["hard_violation_rate"],
    "acceptance": reference["engine_evidence"]["acceptance"],
    "violations": dict(
        zip(
            slices["check_names"].tolist(),
            slices["violations_constrained"].tolist(),
            strict=True,
        )
    ),
}

{'hard_violation_rate': 0.0,
 'acceptance': {'all_hard_checks_pass': True,
  'delta_mae_below_2e-3': True,
  'price_mae_below_1e-3': True},
 'violations': {'price_bounds': 0,
  'put_call_parity': 0,
  'strike_monotonicity': 0,
  'strike_convexity': 0,
  'calendar_monotonicity': 0,
  'spot_monotonicity': 0,
  'nonnegative_gamma': 0,
  'greek_consistency': 0}}

## Heston/COS and Monte Carlo teacher uncertainty

Teacher discrepancies must be interpreted beside standard errors and confidence intervals.

In [7]:
{
    "teacher_ci_coverage_20_seeds": metrics.get(
        "teacher_ci_coverage_20_seeds_by_estimand",
        {"price": metrics["teacher_ci_coverage_20_seeds"]},
    ),
    "teacher_se_ratio_4x_paths": metrics.get(
        "teacher_se_ratio_4x_paths_by_estimand",
        {"price": metrics["teacher_se_ratio_4x_paths"]},
    ),
}

{'teacher_ci_coverage_20_seeds': {'delta': 1.0, 'price': 0.9, 'vega': 0.9},
 'teacher_se_ratio_4x_paths': {'delta': 0.5000675502958665,
  'price': 0.5036523091029843,
  'vega': 0.5018588397313818}}

## Calibration and CPU break-even

In [8]:
{
    "batch_size": slices["batch_size"],
    "analytic_us": slices["analytic_us"],
    "mlp_us": slices["mlp_us"],
    "break_even_batch": metrics["break_even_batch"],
    "calibration_note": "The committed quick reference does not claim calibration coverage.",
}

{'batch_size': array([  1,  32, 256]),
 'analytic_us': array([ 89.967, 153.431, 225.085]),
 'mlp_us': array([   485.275,  59231.713, 126510.266]),
 'break_even_batch': None,
 'calibration_note': 'The committed quick reference does not claim calibration coverage.'}

## Limitations / negative results / research track

Synthetic accuracy is not market predictive power. Failed acceptance checks remain visible in the report.

In [9]:
reference["limitations"]

['Synthetic Black-Scholes accuracy is not evidence of market forecasting power.',
 'The OOD shell remains a stress diagnostic and is not an acceptance claim.',
 'The 512-path Monte-Carlo benchmark is a latency workload, not a precision result.',
 'The small ablation may fail absolute main-run thresholds and is used only for relative comparison.']